# BT-1：Kaggle优先，单平台分段续训
每轮更新恢复包，每5轮保留编号副本；默认约60分钟在完整轮次后暂停。每段结束后先将输出保存为Kaggle私有版本并下载恢复包，核验后才继续；仅写临时盘不算已备份。
只在Kaggle额度不足且上一会话已停止后转Colab，总目标仍为100轮，不从头重训。保存实时模型、EMA、优化器、调度器、随机状态及配置；跨硬件及重建数据加载器不保证逐位相同。


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, base64, subprocess, shutil, time, signal
if not ((3, 11) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError('This locked setup expects Python 3.11-3.12; inspect the runtime before changing versions.')
PLATFORM = 'kaggle' if Path('/kaggle/working').exists() else 'colab'
RESUME_BUNDLE = None  # 已保存且由本项目生成的恢复ZIP绝对路径；首次保持None
PREVIOUS_SESSION_STOPPED = False  # 接续前核验上一会话已停止后改True
if RESUME_BUNDLE and not PREVIOUS_SESSION_STOPPED:
    raise RuntimeError('Confirm previous session stopped before resuming')
if PLATFORM == 'colab' and not RESUME_BUNDLE:
    raise RuntimeError('Kaggle first; Colab only resumes a saved Kaggle checkpoint after quota exhaustion')
WORK_PARENT = Path('/kaggle/temp') if PLATFORM == 'kaggle' else Path('/content')
WORK_PARENT.mkdir(parents=True, exist_ok=True)
WORK = WORK_PARENT / ('bt1_cloud_' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()))
RUN_PARENT = WORK / 'runs'  # 可在首次运行前改为已挂载的持久目录
CLOUD_BATCH = 4             # 启动前可改为1；不在训练途中自动修改
if WORK.exists():
    raise FileExistsError('已有工作目录；请查看已有状态，不要覆盖或重新Run All。')
if shutil.disk_usage(WORK_PARENT).free < 50 * 1024**3:
    raise RuntimeError('Need at least 50 GiB free disk for the proposed setup.')
subprocess.run(['nvidia-smi'], check=True)
WORK.mkdir(); RUN_PARENT.mkdir(parents=True, exist_ok=True)
os.environ['PIP_NO_CACHE_DIR'] = '1'
PY = WORK / 'env/bin/python'
subprocess.run([sys.executable, '-m', 'venv', str(WORK/'env')], check=True)
(WORK/'constraints.txt').write_text('torch==2.7.1\ntorchvision==0.22.1\n')
def run(command, timeout, log_name):
    # External timeout kills this Linux process group, including the training child.
    log = WORK / log_name
    if log.exists():
        raise FileExistsError(log)
    with log.open('xb') as f:
        process = subprocess.Popen([str(x) for x in command], stdout=f, stderr=subprocess.STDOUT, start_new_session=True)
        try:
            returncode = process.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGTERM)
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL); process.wait()
            raise RuntimeError('Timeout; partial outputs retained: ' + str(log))
    print(log.read_text(errors='replace')[-3500:])
    if returncode:
        raise RuntimeError('Step failed; inspect ' + str(log))
run([PY,'-m','pip','install','torch==2.7.1','torchvision==0.22.1','--index-url','https://download.pytorch.org/whl/cu126','--progress-bar','off','--report',WORK/'pip_torch_report.json'], 3600, 'install_torch.log')
run([PY,'-m','pip','install','https://github.com/ultralytics/ultralytics/archive/07958a70205d1388612bd00f8a2f32cf769d8fed.zip','-c',WORK/'constraints.txt','--progress-bar','off','--report',WORK/'pip_ultralytics_report.json'], 1800, 'install_detector.log')
run([PY,'-m','pip','check'], 120, 'pip_check.log')
(WORK/'pip_freeze.txt').write_bytes(subprocess.check_output([str(PY),'-m','pip','freeze']))
print('Independent environment ready:', PY)


In [ ]:
# Exact local source bundle; no credentials.
BUNDLE = {'convert_visdrone_labels.py': {'sha256': '2468a713bc7c975ec99646fef5235992fb4287e2b661937a7ab9bf9bb36eb25e', 'base64': 'IiIiU2luZ2xlLWZpbGUgVmlzRHJvbmUgbGFiZWwgYWRhcHRlcjsgbm8gaW1hZ2VzLCBtb2RlbCBpbXBvcnRzIG9yIGJ1bGsgY29udmVyc2lvbi4KCkRyb3BwaW5nIHBvc2l0aXZlIGxhYmVscyBkb2VzIE5PVCBpbXBsZW1lbnQgcmVnaW9uLW5ldXRyYWwgdHJhaW5pbmcgaWdub3JlLgpUaGUgQ0xJIHJlcXVpcmVzIGFuIHVudXNlZCBvdXRwdXQgZGlyZWN0b3J5IHVuZGVyIHRoaXMgcmVwb3NpdG9yeSdzIHByb2Nlc3NlZApWaXNEcm9uZSBkaXJlY3RvcnkuIE9yaWdpbmFsIGFubm90YXRpb25zIGFyZSByZWFkIG9ubHkuIFNlZSBMYWJlbF9BZGFwdGVyX0NoZWNrLm1kLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQpmcm9tIGRlY2ltYWwgaW1wb3J0IERlY2ltYWwsIEludmFsaWRPcGVyYXRpb24sIGxvY2FsY29udGV4dAppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClZFUlNJT04gPSAibGFiZWwtYWRhcHRlci0wLjIiClJFUE8gPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1syXQpPVVRQVVRfUk9PVCA9IFJFUE8gLyAiMTFfRGF0YXNldHMvcHJvY2Vzc2VkL1Zpc0Ryb25lIgpOQU1FUyA9ICgKICAgICJwZWRlc3RyaWFuIiwgInBlb3BsZSIsICJiaWN5Y2xlIiwgImNhciIsICJ2YW4iLCAidHJ1Y2siLCAidHJpY3ljbGUiLAogICAgImF3bmluZy10cmljeWNsZSIsICJidXMiLCAibW90b3IiLAopCgoKY2xhc3MgQW5ub3RhdGlvbkVycm9yKFZhbHVlRXJyb3IpOgogICAgIiIiSW5wdXQgY2Fubm90IGJlIGNvbnZlcnRlZCB3aXRob3V0IGFuIHVuYXBwcm92ZWQgcmVwYWlyLiIiIgoKCmRlZiBjb252ZXJ0X3RleHQodGV4dDogc3RyLCB3aWR0aDogaW50LCBoZWlnaHQ6IGludCkgLT4gdHVwbGVbc3RyLCBsaXN0W2RpY3RdXToKICAgICIiIlZhbGlkYXRlIGFsbCByb3dzIGJlZm9yZSByZXR1cm5pbmcgbGFiZWxzIGFuZCBhIGNvbXBsZXRlIHJvdyBhdWRpdC4KCiAgICBCbGFuayBmaWxlcyBhcmUgdmFsaWQ7IGJsYW5rIHJvd3MgaW5zaWRlIGEgbm9uYmxhbmsgZmlsZSBhcmUgbWFsZm9ybWVkLgogICAgRmxhZ3MgbXVzdCBiZSBpbnRlZ2Vyczsgc2NvcmUgaXMgcmVzdHJpY3RlZCB0byAwLzEgYW5kIGNhdGVnb3J5IHRvIDAuLjExLgogICAgT2NjbHVzaW9uL3RydW5jYXRpb24gdmFsdWVzIGFyZSBwcmVzZXJ2ZWQsIG5vdCB1c2VkIHRvIGZpbHRlciBwb3NpdGl2ZXMuCiAgICAiIiIKICAgIGlmIGFueSh0eXBlKHYpIGlzIG5vdCBpbnQgb3IgdiA8PSAwIGZvciB2IGluICh3aWR0aCwgaGVpZ2h0KSk6CiAgICAgICAgcmFpc2UgQW5ub3RhdGlvbkVycm9yKCJJbWFnZSBkaW1lbnNpb25zIG11c3QgYmUgcG9zaXRpdmUgaW50ZWdlcnMiKQogICAgaWYgbm90IHRleHQuc3RyaXAoKToKICAgICAgICByZXR1cm4gIiIsIFtdCiAgICByb3dzLCBvdXRwdXQgPSBbXSwgW10KICAgIHdpdGggbG9jYWxjb250ZXh0KCkgYXMgY3R4OgogICAgICAgIGN0eC5wcmVjID0gNTAKICAgICAgICBXLCBIID0gRGVjaW1hbCh3aWR0aCksIERlY2ltYWwoaGVpZ2h0KQogICAgICAgIGZvciBudW1iZXIsIHJhdyBpbiBlbnVtZXJhdGUodGV4dC5zcGxpdGxpbmVzKCksIDEpOgogICAgICAgICAgICBmaWVsZHMgPSByYXcuc3BsaXQoIiwiKQogICAgICAgICAgICAjIE1hdGNoIHRoZSBleGlzdGluZyBkYXRhc2V0IGF1ZGl0b3IncyBvcHRpb25hbCB0ZXJtaW5hbCBkZWxpbWl0ZXIgcnVsZS4KICAgICAgICAgICAgIyBPcmlnaW5hbCByYXcgdGV4dCByZW1haW5zIGluIHRoZSByb3cgYXVkaXQ7IG5vIG51bWVyaWMgZmllbGQgaXMgZGlzY2FyZGVkLgogICAgICAgICAgICBpZiBsZW4oZmllbGRzKSA9PSA5IGFuZCBub3QgZmllbGRzWy0xXS5zdHJpcCgpOgogICAgICAgICAgICAgICAgZmllbGRzLnBvcCgpCiAgICAgICAgICAgIGlmIGxlbihmaWVsZHMpICE9IDggb3IgYW55KG5vdCB2LnN0cmlwKCkgZm9yIHYgaW4gZmllbGRzKToKICAgICAgICAgICAgICAgIHJhaXNlIEFubm90YXRpb25FcnJvcihmIkxpbmUge251bWJlcn06IGV4cGVjdGVkIGVpZ2h0IG5vbmVtcHR5IGZpZWxkcyIpCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHZhbHVlcyA9IFtEZWNpbWFsKHYuc3RyaXAoKSkgZm9yIHYgaW4gZmllbGRzXQogICAgICAgICAgICBleGNlcHQgSW52YWxpZE9wZXJhdGlvbiBhcyBleGM6CiAgICAgICAgICAgICAgICByYWlzZSBBbm5vdGF0aW9uRXJyb3IoZiJMaW5lIHtudW1iZXJ9OiBpbnZhbGlkIG51bWVyaWMgZmllbGQiKSBmcm9tIGV4YwogICAgICAgICAgICBpZiBub3QgYWxsKHYuaXNfZmluaXRlKCkgZm9yIHYgaW4gdmFsdWVzKToKICAgICAgICAgICAgICAgIHJhaXNlIEFubm90YXRpb25FcnJvcihmIkxpbmUge251bWJlcn06IG5vbmZpbml0ZSB2YWx1ZSIpCiAgICAgICAgICAgIGlmIGFueSh2ICE9IHYudG9faW50ZWdyYWxfdmFsdWUoKSBmb3IgdiBpbiB2YWx1ZXNbNDpdKToKICAgICAgICAgICAgICAgIHJhaXNlIEFubm90YXRpb25FcnJvcihmIkxpbmUge251bWJlcn06IG5vbmludGVnZXIgY2F0ZWdvcnkvZmxhZyIpCiAgICAgICAgICAgIHgsIHksIHcsIGggPSB2YWx1ZXNbOjRdCiAgICAgICAgICAgIHNjb3JlLCBjYXRlZ29yeSwgdHJ1bmNhdGlvbiwgb2NjbHVzaW9uID0gbWFwKGludCwgdmFsdWVzWzQ6XSkKICAgICAgICAgICAgaWYgc2NvcmUgbm90IGluICgwLCAxKSBvciBub3QgMCA8PSBjYXRlZ29yeSA8PSAxMToKICAgICAgICAgICAgICAgIHJhaXNlIEFubm90YXRpb25FcnJvcihmIkxpbmUge251bWJlcn06IHNjb3JlL2NhdGVnb3J5IG91dHNpZGUgYWxsb3dlZCByYW5nZSIpCiAgICAgICAgICAgIGlmICh4IDwgMCBvciB5IDwgMCBvciB4ID4gVyBvciB5ID4gSAogICAgICAgICAgICAgICAgICAgIG9yICh3ID4gMCBhbmQgeCArIHcgPiBXKSBvciAoaCA+IDAgYW5kIHkgKyBoID4gSCkpOgogICAgICAgICAgICAgICAgcmFpc2UgQW5ub3RhdGlvbkVycm9yKGYiTGluZSB7bnVtYmVyfTogZ2VvbWV0cnkgb3V0c2lkZSBpbWFnZSBib3VuZHMiKQogICAgICAgICAgICByZWFzb25zID0gW10KICAgICAgICAgICAgaWYgc2NvcmUgPT0gMDoKICAgICAgICAgICAgICAgIHJlYXNvbnMuYXBwZW5kKCJzY29yZV96ZXJvIikKICAgICAgICAgICAgaWYgY2F0ZWdvcnkgPT0gMDoKICAgICAgICAgICAgICAgIHJlYXNvbnMuYXBwZW5kKCJpZ25vcmVfY2F0ZWdvcnkiKQogICAgICAgICAgICBpZiBjYXRlZ29yeSA9PSAxMToKICAgICAgICAgICAgICAgIHJlYXNvbnMuYXBwZW5kKCJvdGhlcnNfY2F0ZWdvcnkiKQogICAgICAgICAgICBpZiB3IDw9IDAgb3IgaCA8PSAwOgogICAgICAgICAgICAgICAgcmVhc29ucy5hcHBlbmQoIm5vbnBvc2l0aXZlX3NpemUiKQogICAgICAgICAgICByb3cgPSBkaWN0KGxpbmU9bnVtYmVyLCByYXc9cmF3LCBzY29yZT1zY29yZSwgY2F0ZWdvcnk9Y2F0ZWdvcnksCiAgICAgICAgICAgICAgICAgICAgICAgdHJ1bmNhdGlvbj10cnVuY2F0aW9uLCBvY2NsdXNpb249b2NjbHVzaW9uLAogICAgICAgICAgICAgICAgICAgICAgIHJlYXNvbnM9cmVhc29ucywgYWN0aW9uPSJleGNsdWRlZCIgaWYgcmVhc29ucyBlbHNlICJrZXB0IikKICAgICAgICAgICAgaWYgbm90IHJlYXNvbnM6CiAgICAgICAgICAgICAgICBub3JtYWxpemVkID0gKCh4ICsgdyAvIDIpIC8gVywgKHkgKyBoIC8gMikgLyBILCB3IC8gVywgaCAvIEgpCiAgICAgICAgICAgICAgICB0b2tlbnMgPSBbZm9ybWF0KGZsb2F0KHYpLCAiLjE3ZyIpIGZvciB2IGluIG5vcm1hbGl6ZWRdCiAgICAgICAgICAgICAgICBwYXJzZWQgPSBbZmxvYXQodikgZm9yIHYgaW4gdG9rZW5zXQogICAgICAgICAgICAgICAgaWYgbm90IGFsbChtYXRoLmlzZmluaXRlKHYpIGFuZCAwIDwgdiA8PSAxIGZvciB2IGluIHBhcnNlZCk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgQW5ub3RhdGlvbkVycm9yKGYiTGluZSB7bnVtYmVyfTogbm9ybWFsaXphdGlvbiB1bmRlcmZsb3cvcmFuZ2UgZXJyb3IiKQogICAgICAgICAgICAgICAgbGFiZWwgPSBmIntjYXRlZ29yeSAtIDF9ICIgKyAiICIuam9pbih0b2tlbnMpCiAgICAgICAgICAgICAgICBvdXRwdXQuYXBwZW5kKGxhYmVsKQogICAgICAgICAgICAgICAgcm93LnVwZGF0ZShvdXRwdXRfbGluZT1sZW4ob3V0cHV0KSwgbGFiZWw9bGFiZWwpCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHJvdykKICAgIHJldHVybiAiIi5qb2luKGxpbmUgKyAiXG4iIGZvciBsaW5lIGluIG91dHB1dCksIHJvd3MKCgpkZWYgY29udmVydF9maWxlKHNvdXJjZTogUGF0aCwgd2lkdGg6IGludCwgaGVpZ2h0OiBpbnQsIGRlc3RpbmF0aW9uOiBQYXRoKSAtPiBkaWN0OgogICAgIiIiQ3JlYXRlIGxhYmVscy50eHQgKyBhdWRpdC5qc29uIGluIGEgZnJlc2ggZGVyaXZlZCBkaXJlY3RvcnkuCgogICAgSW5wdXQgZXJyb3JzIGNyZWF0ZSBubyBkZXN0aW5hdGlvbi4gSS9PIGZhaWx1cmVzIG1heSBsZWF2ZSBhbiBpbmNvbXBsZXRlCiAgICBkaXJlY3RvcnksIHdoaWNoIGlzIHJldGFpbmVkIGFuZCBjYW5ub3QgYmUgb3ZlcndyaXR0ZW4gYnkgYSByZXRyeS4KICAgIGF1ZGl0Lmpzb24gd2l0aCBzdGF0dXM9Y29tcGxldGUgaXMgd3JpdHRlbiBsYXN0LgogICAgIiIiCiAgICBzb3VyY2UsIGRlc3RpbmF0aW9uID0gUGF0aChzb3VyY2UpLnJlc29sdmUoKSwgUGF0aChkZXN0aW5hdGlvbikucmVzb2x2ZSgpCiAgICBhbGxvd2VkID0gT1VUUFVUX1JPT1QucmVzb2x2ZSgpCiAgICBpZiBkZXN0aW5hdGlvbiA9PSBhbGxvd2VkIG9yIG5vdCBkZXN0aW5hdGlvbi5pc19yZWxhdGl2ZV90byhhbGxvd2VkKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPdXRwdXQgbXVzdCBiZSBhIG5ldyBzdWJkaXJlY3Rvcnkgb2YgcHJvY2Vzc2VkL1Zpc0Ryb25lIikKICAgIGlmIGRlc3RpbmF0aW9uLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcihmIlJlZnVzaW5nIGV4aXN0aW5nIG91dHB1dDoge2Rlc3RpbmF0aW9ufSIpCiAgICBpZiBzb3VyY2UuaXNfcmVsYXRpdmVfdG8oZGVzdGluYXRpb24pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk91dHB1dCBjYW5ub3QgY29udGFpbiB0aGUgaW5wdXQiKQogICAgb3JpZ2luYWwgPSBzb3VyY2UucmVhZF9ieXRlcygpICAjIE1pc3NpbmcgYW5ub3RhdGlvbnMgYXJlIGVycm9ycywgbmV2ZXIgZW1wdHkgbGFiZWxzLgogICAgbGFiZWxzLCByb3dzID0gY29udmVydF90ZXh0KG9yaWdpbmFsLmRlY29kZSgidXRmLTgiKSwgd2lkdGgsIGhlaWdodCkKICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KG9yaWdpbmFsKS5oZXhkaWdlc3QoKQogICAgaWYgc291cmNlLnJlYWRfYnl0ZXMoKSAhPSBvcmlnaW5hbDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlNvdXJjZSBjaGFuZ2VkIGR1cmluZyBjb252ZXJzaW9uOyBubyBvdXRwdXQgd3JpdHRlbiIpCiAgICBlbmNvZGVkID0gbGFiZWxzLmVuY29kZSgidXRmLTgiKQogICAgcmVwb3J0ID0gZGljdCgKICAgICAgICBzdGF0dXM9ImNvbXBsZXRlIiwgYWRhcHRlcl92ZXJzaW9uPVZFUlNJT04sCiAgICAgICAgYWRhcHRlcl9zaGEyNTY9aGFzaGxpYi5zaGEyNTYoUGF0aChfX2ZpbGVfXykucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICBzb3VyY2U9c3RyKHNvdXJjZSksIHNvdXJjZV9zaGEyNTY9ZGlnZXN0LCB3aWR0aD13aWR0aCwgaGVpZ2h0PWhlaWdodCwKICAgICAgICBuYW1lcz1saXN0KE5BTUVTKSwgaW5wdXRfcm93cz1sZW4ocm93cyksIGtlcHRfcm93cz1zdW0oclsiYWN0aW9uIl0gPT0gImtlcHQiIGZvciByIGluIHJvd3MpLAogICAgICAgIGVtcHR5X29yaWdpbj0oInNvdXJjZV9lbXB0eSIgaWYgbm90IHJvd3MgZWxzZSAiYWxsX2V4Y2x1ZGVkIiBpZiBub3QgbGFiZWxzIGVsc2UgTm9uZSksCiAgICAgICAgdHJhaW5pbmdfcmVnaW9uX2lnbm9yZT1GYWxzZSwgb3V0cHV0X3NoYTI1Nj1oYXNobGliLnNoYTI1NihlbmNvZGVkKS5oZXhkaWdlc3QoKSwKICAgICAgICBjb29yZGluYXRlX3J1bGU9Inh5d2ggcGl4ZWxzIC0+IG5vcm1hbGl6ZWQgY2VudGVyIHh5d2g7IG5vICsvLTE7IGZsb2F0IC4xN2ciLAogICAgICAgIHJvd3M9cm93cywKICAgICkKICAgIGRlc3RpbmF0aW9uLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9RmFsc2UpCiAgICB3aXRoIChkZXN0aW5hdGlvbiAvICJsYWJlbHMudHh0Iikub3BlbigieGIiKSBhcyBoYW5kbGU6CiAgICAgICAgaGFuZGxlLndyaXRlKGVuY29kZWQpCiAgICB3aXRoIChkZXN0aW5hdGlvbiAvICJhdWRpdC5qc29uIikub3BlbigieCIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IlxuIikgYXMgaGFuZGxlOgogICAgICAgIGpzb24uZHVtcChyZXBvcnQsIGhhbmRsZSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MikKICAgICAgICBoYW5kbGUud3JpdGUoIlxuIikKICAgIHJldHVybiByZXBvcnQKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgic291cmNlIiwgdHlwZT1QYXRoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13aWR0aCIsIHR5cGU9aW50LCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1oZWlnaHQiLCB0eXBlPWludCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHRyeToKICAgICAgICByZXBvcnQgPSBjb252ZXJ0X2ZpbGUoYXJncy5zb3VyY2UsIGFyZ3Mud2lkdGgsIGFyZ3MuaGVpZ2h0LCBhcmdzLm91dHB1dCkKICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgT1NFcnJvciwgUnVudGltZUVycm9yKSBhcyBleGM6CiAgICAgICAgcGFyc2VyLmV4aXQoMiwgZiJDb252ZXJzaW9uIHN0b3BwZWQ6IHtleGN9XG4iKQogICAgcHJpbnQoanNvbi5kdW1wcyh7a2V5OiByZXBvcnRba2V5XSBmb3Iga2V5IGluICgic3RhdHVzIiwgImlucHV0X3Jvd3MiLCAia2VwdF9yb3dzIiwgImVtcHR5X29yaWdpbiIpfSkpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='}, 'prepare_bt1_data.py': {'sha256': 'd10242c538427e6bde97439bba3b9e5b43457f229704ac23a8032007a9c0aa0f', 'base64': 'IiIiUHJlcGFyZSBmcmVzaCBCVC0xIGlucHV0cyBmcm9tIHR3byBmaXhlZCBaSVBzOyBuZXZlciByZWFkcyB0ZXN0LWRldi4iIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBoYXNobGliCmltcG9ydCBpbwppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgsIFB1cmVQb3NpeFBhdGgKaW1wb3J0IHppcGZpbGUKCmZyb20gUElMIGltcG9ydCBJbWFnZQpmcm9tIGNvbnZlcnRfdmlzZHJvbmVfbGFiZWxzIGltcG9ydCBjb252ZXJ0X3RleHQsIE5BTUVTCgpTSEEgPSB7InRyYWluIjogIjg2YTc3ZWJhOTMxMzdiZmMxNmU0OTkzODYwZGU5MjQ1YjA2NzVjMGRiYTBkM2FiOThmYjQ1ODY5OWUyNTZmODQiLAogICAgICAgInZhbCI6ICJhYmVlYTA2MzAzN2U1ZDIwMzk4ODM3ZGViMTEwODRlNjUyNDAyYTM0ZGRmNGYyMDdiZGY1NDFhNmYyYTM1ZWY5In0KCgpkZWYgZGlnZXN0KHBhdGgpOgogICAgd2l0aCBQYXRoKHBhdGgpLm9wZW4oInJiIikgYXMgZjoKICAgICAgICByZXR1cm4gaGFzaGxpYi5maWxlX2RpZ2VzdChmLCAic2hhMjU2IikuaGV4ZGlnZXN0KCkKCgpkZWYgcHJlcGFyZSh0cmFpbl96aXAsIHZhbF96aXAsIG91dCwgc21va2U9RmFsc2UpOgogICAgb3V0ID0gUGF0aChvdXQpLnJlc29sdmUoKQogICAgaWYgb3V0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcihmIlJlZnVzaW5nIGV4aXN0aW5nIG91dHB1dDoge291dH0iKQogICAgZm9yIHNwbGl0LCBwYXRoIGluIFsoInRyYWluIiwgdHJhaW5femlwKSwgKCJ2YWwiLCB2YWxfemlwKV06CiAgICAgICAgaWYgZGlnZXN0KHBhdGgpICE9IFNIQVtzcGxpdF06CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7c3BsaXR9IFpJUCBpZGVudGl0eSBkaWZmZXJzIGZyb20gYXBwcm92ZWQgaW5wdXQiKQogICAgb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgIG1hbmlmZXN0cywgY291bnRzID0ge30sIHt9CiAgICB3aXRoIChvdXQgLyAiY29udmVyc2lvbi5qc29ubCIpLm9wZW4oIngiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBhdWRpdDoKICAgICAgICBmb3Igc3BsaXQsIHNvdXJjZSBpbiBbKCJ0cmFpbiIsIHRyYWluX3ppcCksICgidmFsIiwgdmFsX3ppcCldOgogICAgICAgICAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShzb3VyY2UpIGFzIHo6CiAgICAgICAgICAgICAgICBuYW1lcyA9IHoubmFtZWxpc3QoKQogICAgICAgICAgICAgICAgaWYgbGVuKHNldChuYW1lcykpICE9IGxlbihuYW1lcyk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRHVwbGljYXRlIFpJUCBtZW1iZXJzIikKICAgICAgICAgICAgICAgIGZvciBuIGluIG5hbWVzOgogICAgICAgICAgICAgICAgICAgIHBwID0gUHVyZVBvc2l4UGF0aChuKQogICAgICAgICAgICAgICAgICAgIGlmIHBwLmlzX2Fic29sdXRlKCkgb3IgIi4uIiBpbiBwcC5wYXJ0cyBvciAiXFwiIGluIG4gb3IgIjoiIGluIG46CiAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuc2FmZSBaSVAgcGF0aCIpCiAgICAgICAgICAgICAgICBwcmVmaXggPSBmIlZpc0Ryb25lMjAxOS1ERVQte3NwbGl0fSIKICAgICAgICAgICAgICAgIGltYWdlcyA9IHNvcnRlZChuIGZvciBuIGluIG5hbWVzIGlmIG4uc3RhcnRzd2l0aChwcmVmaXggKyAiL2ltYWdlcy8iKSBhbmQgbi5lbmRzd2l0aCgiLmpwZyIpKQogICAgICAgICAgICAgICAgZXhwZWN0ZWQgPSA2NDcxIGlmIHNwbGl0ID09ICJ0cmFpbiIgZWxzZSA1NDgKICAgICAgICAgICAgICAgIGlmIGxlbihpbWFnZXMpICE9IGV4cGVjdGVkOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbmV4cGVjdGVkIHtzcGxpdH0gaW1hZ2UgY291bnQ6IHtsZW4oaW1hZ2VzKX0iKQogICAgICAgICAgICAgICAgaWYgc3BsaXQgPT0gInZhbCI6CiAgICAgICAgICAgICAgICAgICAgaW1hZ2VzLnNvcnQoa2V5PWxhbWJkYSBuOiAoaGFzaGxpYi5zaGEyNTYoKCJkaWFnLXYxOiIgKyBQdXJlUG9zaXhQYXRoKG4pLm5hbWUpLmVuY29kZSgpKS5oZXhkaWdlc3QoKSwgUHVyZVBvc2l4UGF0aChuKS5uYW1lKSkKICAgICAgICAgICAgICAgICAgICAjIFRoZSByZW1haW5pbmcgNTAwIGltYWdlcyBhcmUgbm90IGRlY29kZWQgb3IgbGFiZWxsZWQgaGVyZS4KICAgICAgICAgICAgICAgICAgICAob3V0IC8gImRpYWc1MDBfbmFtZXMudHh0Iikud3JpdGVfdGV4dCgiXG4iLmpvaW4oUHVyZVBvc2l4UGF0aChuKS5uYW1lIGZvciBuIGluIGltYWdlc1s0ODpdKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgICAgICAgICAgICAgaW1hZ2VzID0gaW1hZ2VzWzo0OF0KICAgICAgICAgICAgICAgIGVsaWYgc21va2U6CiAgICAgICAgICAgICAgICAgICAgaW1hZ2VzID0gaW1hZ2VzWzo0XQogICAgICAgICAgICAgICAgdGFnID0gInRyYWluIiBpZiBzcGxpdCA9PSAidHJhaW4iIGVsc2UgImNhbDQ4IgogICAgICAgICAgICAgICAgcGF0aHMgPSBbXQogICAgICAgICAgICAgICAgdG90YWwgPSBrZXB0ID0gZXhjbHVkZWQgPSAwCiAgICAgICAgICAgICAgICBmb3Igc3ViIGluIFsiaW1hZ2VzIiwgImxhYmVscyIsICJhbm5vdGF0aW9ucyJdOgogICAgICAgICAgICAgICAgICAgIChvdXQgLyBzdWIgLyB0YWcpLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgICAgICAgICAgICAgIGZvciBuIGluIGltYWdlczoKICAgICAgICAgICAgICAgICAgICBzdGVtID0gUHVyZVBvc2l4UGF0aChuKS5zdGVtCiAgICAgICAgICAgICAgICAgICAgYW5uID0gcHJlZml4ICsgIi9hbm5vdGF0aW9ucy8iICsgc3RlbSArICIudHh0IgogICAgICAgICAgICAgICAgICAgIHJhdyA9IHoucmVhZChhbm4pICAjIE1pc3NpbmcgbGFiZWwgaXMgYW4gZXJyb3IsIG5vdCBhbiBlbXB0eSBsYWJlbC4KICAgICAgICAgICAgICAgICAgICBpbWcgPSB6LnJlYWQobikKICAgICAgICAgICAgICAgICAgICB3aXRoIEltYWdlLm9wZW4oaW8uQnl0ZXNJTyhpbWcpKSBhcyBpbToKICAgICAgICAgICAgICAgICAgICAgICAgdywgaCA9IGltLnNpemUKICAgICAgICAgICAgICAgICAgICAgICAgaW0udmVyaWZ5KCkKICAgICAgICAgICAgICAgICAgICBsYWJlbHMsIHJvd3MgPSBjb252ZXJ0X3RleHQocmF3LmRlY29kZSgidXRmLTgiKSwgdywgaCkKICAgICAgICAgICAgICAgICAgICBpbWFnZV9wYXRoID0gb3V0IC8gImltYWdlcyIgLyB0YWcgLyAoc3RlbSArICIuanBnIikKICAgICAgICAgICAgICAgICAgICBpbWFnZV9wYXRoLndyaXRlX2J5dGVzKGltZykKICAgICAgICAgICAgICAgICAgICAob3V0IC8gImFubm90YXRpb25zIiAvIHRhZyAvIChzdGVtICsgIi50eHQiKSkud3JpdGVfYnl0ZXMocmF3KQogICAgICAgICAgICAgICAgICAgIChvdXQgLyAibGFiZWxzIiAvIHRhZyAvIChzdGVtICsgIi50eHQiKSkud3JpdGVfdGV4dChsYWJlbHMsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IlxuIikKICAgICAgICAgICAgICAgICAgICBwYXRocy5hcHBlbmQoaW1hZ2VfcGF0aC5hc19wb3NpeCgpKQogICAgICAgICAgICAgICAgICAgIG5fa2VwdCA9IHN1bShyWyJhY3Rpb24iXSA9PSAia2VwdCIgZm9yIHIgaW4gcm93cykKICAgICAgICAgICAgICAgICAgICB0b3RhbCArPSBsZW4ocm93cyk7IGtlcHQgKz0gbl9rZXB0OyBleGNsdWRlZCArPSBsZW4ocm93cykgLSBuX2tlcHQKICAgICAgICAgICAgICAgICAgICBhdWRpdC53cml0ZShqc29uLmR1bXBzKGRpY3Qoc3BsaXQ9dGFnLCBtZW1iZXI9biwgd2lkdGg9dywgaGVpZ2h0PWgsCiAgICAgICAgICAgICAgICAgICAgICAgIGltYWdlX3NoYTI1Nj1oYXNobGliLnNoYTI1NihpbWcpLmhleGRpZ2VzdCgpLCBhbm5vdGF0aW9uX3NoYTI1Nj1oYXNobGliLnNoYTI1NihyYXcpLmhleGRpZ2VzdCgpLAogICAgICAgICAgICAgICAgICAgICAgICBsYWJlbF9zaGEyNTY9aGFzaGxpYi5zaGEyNTYobGFiZWxzLmVuY29kZSgpKS5oZXhkaWdlc3QoKSwgcm93cz1yb3dzKSwgZW5zdXJlX2FzY2lpPUZhbHNlKSArICJcbiIpCiAgICAgICAgICAgICAgICBtYW5pZmVzdCA9IG91dCAvICh0YWcgKyAiLnR4dCIpCiAgICAgICAgICAgICAgICBtYW5pZmVzdC53cml0ZV90ZXh0KCJcbiIuam9pbihwYXRocykgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgICAgICAgICAgbWFuaWZlc3RzW3RhZ10gPSBzdHIobWFuaWZlc3QpCiAgICAgICAgICAgICAgICBjb3VudHNbdGFnXSA9IGRpY3QoaW1hZ2VzPWxlbihwYXRocyksIGlucHV0X3Jvd3M9dG90YWwsIGtlcHRfcm93cz1rZXB0LCBleGNsdWRlZF9yb3dzPWV4Y2x1ZGVkKQogICAgICAgICAgICAgICAgYXNzZXJ0IHRvdGFsID09IGtlcHQgKyBleGNsdWRlZAogICAgICAgICMgSlNPTiBpcyB2YWxpZCBZQU1MOyBleHRlbnNpb24gY2hvc2VuIGZvciB0aGUgZGV0ZWN0aW9uIGZyYW1ld29yay4KICAgICAgICBkYXRhID0gZGljdChwYXRoPXN0cihvdXQpLCB0cmFpbj1tYW5pZmVzdHNbInRyYWluIl0sIHZhbD1tYW5pZmVzdHNbImNhbDQ4Il0sIG5jPTEwLCBuYW1lcz1saXN0KE5BTUVTKSkKICAgICAgICAob3V0IC8gImRhdGEueWFtbCIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhkYXRhLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXBvcnQgPSBkaWN0KHN0YXR1cz0iQ09NUExFVEUiLCBzbW9rZT1zbW9rZSwgc291cmNlX3NoYTI1Nj1TSEEsIGNvdW50cz1jb3VudHMsCiAgICAgICAgICAgICAgICAgIHRyYWluaW5nX3JlZ2lvbl9pZ25vcmU9RmFsc2UsIG1hbmlmZXN0X2hhc2hlcz17azogZGlnZXN0KHYpIGZvciBrLCB2IGluIG1hbmlmZXN0cy5pdGVtcygpfSwKICAgICAgICAgICAgICAgICAgYWRhcHRlcl9zaGEyNTY9ZGlnZXN0KFBhdGgoX19maWxlX18pLndpdGhfbmFtZSgiY29udmVydF92aXNkcm9uZV9sYWJlbHMucHkiKSkpCiAgICAob3V0IC8gInByZXBhcmF0aW9uLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludChqc29uLmR1bXBzKHJlcG9ydCksIGZsdXNoPVRydWUpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwLmFkZF9hcmd1bWVudCgiLS10cmFpbi16aXAiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS12YWwtemlwIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcC5hZGRfYXJndW1lbnQoIi0tb3V0cHV0IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc21va2UiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYSA9IHAucGFyc2VfYXJncygpCiAgICBwcmVwYXJlKGEudHJhaW5femlwLCBhLnZhbF96aXAsIGEub3V0cHV0LCBhLnNtb2tlKQo='}, 'run_bt1.py': {'sha256': 'a3f12deb33defb29761a6b60320c82713db069b7f33e5e1a14fa7ff4feb357fc', 'base64': '...'}, 'bt1_checkpoint.py': {'sha256': '3cf89f343d6c8c5f4d6850faf83dc793ec66e41aeff2fa3ade3bb124eabb7b24', 'base64': 'IiIiUG9ydGFibGUgZXBvY2gtYm91bmRhcnkgcmVjb3Zlcnk7IGRvZXMgbm90IGNsYWltIGJpdHdpc2UgY3Jvc3MtR1BVIHJlcGxheS4iIiIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCByYW5kb20KaW1wb3J0IHNodXRpbAppbXBvcnQgemlwZmlsZQoKCmRlZiBzaGEocGF0aCk6CiAgICB3aXRoIFBhdGgocGF0aCkub3BlbigncmInKSBhcyBmOgogICAgICAgIHJldHVybiBoYXNobGliLmZpbGVfZGlnZXN0KGYsICdzaGEyNTYnKS5oZXhkaWdlc3QoKQoKCmRlZiBjb250cmFjdChzcGVjLCBiYXRjaCwgcHJlcGFyYXRpb24pOgogICAgYXJncyA9IHNwZWNbJ3RyYWluX2FyZ3MnXS5jb3B5KCkKICAgIGFyZ3NbJ2JhdGNoJ10gPSBiYXRjaAogICAgYXJnc1sncmVzdW1lJ10gPSBGYWxzZQogICAgcmV0dXJuIGRpY3QoYXJncz1hcmdzLCBkYXRhPXNwZWNbJ2RhdGEnXSwgY29tbWl0PXNwZWNbJ3VsdHJhbHl0aWNzX2NvbW1pdCddLAogICAgICAgICAgICAgICAgaW5pdGlhbGl6YXRpb249c3BlY1snaW5pdGlhbGl6YXRpb24nXVsnc2hhMjU2J10sCiAgICAgICAgICAgICAgICBjb252ZXJzaW9uX3NoYTI1Nj1zaGEoUGF0aChwcmVwYXJhdGlvbikucGFyZW50Lydjb252ZXJzaW9uLmpzb25sJykpCgoKZGVmIHdyaXRlX3JlY292ZXJ5KHRyYWluZXIsIG91dHB1dCwgaWRlbnRpdHkpOgogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBpbXBvcnQgdG9yY2gKICAgIGZyb20gY29weSBpbXBvcnQgZGVlcGNvcHkKICAgIGZyb20gdWx0cmFseXRpY3MudXRpbHMudG9yY2hfdXRpbHMgaW1wb3J0IHVud3JhcF9tb2RlbAogICAgb3V0cHV0ID0gUGF0aChvdXRwdXQpCiAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZXBvY2ggPSB0cmFpbmVyLmVwb2NoICsgMQogICAgIyBOYXRpdmUgY2hlY2twb2ludHMgY29udGFpbiBoYWxmLXByZWNpc2lvbiBFTUEuIFByZXNlcnZlIGxpdmUgRlAzMiBzdGF0ZSBzZXBhcmF0ZWx5LgogICAgcGF5bG9hZCA9IGRpY3QoZXBvY2g9dHJhaW5lci5lcG9jaCwgYmVzdF9maXRuZXNzPXRyYWluZXIuYmVzdF9maXRuZXNzLAogICAgICAgIG1vZGVsPU5vbmUsIGVtYT1kZWVwY29weSh1bndyYXBfbW9kZWwodHJhaW5lci5lbWEuZW1hKSkuY3B1KCkuZmxvYXQoKSwKICAgICAgICB1cGRhdGVzPXRyYWluZXIuZW1hLnVwZGF0ZXMsIG9wdGltaXplcj1kZWVwY29weSh0cmFpbmVyLm9wdGltaXplci5zdGF0ZV9kaWN0KCkpLAogICAgICAgIHNjYWxlcj10cmFpbmVyLnNjYWxlci5zdGF0ZV9kaWN0KCksIHRyYWluX2FyZ3M9dmFycyh0cmFpbmVyLmFyZ3MpLmNvcHkoKSwKICAgICAgICB2ZXJzaW9uPSc4LjQuOTAnLCBidDFfZm9ybWF0PTEsIGJ0MV9jb250cmFjdD1pZGVudGl0eSwKICAgICAgICBidDFfbGl2ZT17azp2LmRldGFjaCgpLmNwdSgpLmNsb25lKCkgZm9yIGssdiBpbiB1bndyYXBfbW9kZWwodHJhaW5lci5tb2RlbCkuc3RhdGVfZGljdCgpLml0ZW1zKCl9LAogICAgICAgIGJ0MV9zY2hlZHVsZXI9dHJhaW5lci5zY2hlZHVsZXIuc3RhdGVfZGljdCgpLAogICAgICAgIGJ0MV9lbGFwc2VkX3NlY29uZHM9Z2V0YXR0cih0cmFpbmVyLCAnX2J0MV9lbGFwc2VkX3NlY29uZHMnLCAwKSwKICAgICAgICBidDFfcm5nPWRpY3QocHl0aG9uPXJhbmRvbS5nZXRzdGF0ZSgpLCBudW1weT1ucC5yYW5kb20uZ2V0X3N0YXRlKCksCiAgICAgICAgICAgICAgICAgICAgIHRvcmNoPXRvcmNoLmdldF9ybmdfc3RhdGUoKSwgY3VkYT10b3JjaC5jdWRhLmdldF9ybmdfc3RhdGUoMCkpKQogICAgdG1wID0gb3V0cHV0LydyZXN1bWUucHQudG1wJwogICAgdG9yY2guc2F2ZShwYXlsb2FkLCB0bXApCiAgICB0bXAucmVwbGFjZShvdXRwdXQvJ3Jlc3VtZS5wdCcpCiAgICBtZXRhID0gZGljdChmb3JtYXQ9MSwgY29tcGxldGVkX2Vwb2Nocz1lcG9jaCwgdG90YWxfZXBvY2hzPXRyYWluZXIuYXJncy5lcG9jaHMsCiAgICAgICAgY2hlY2twb2ludF9zaGEyNTY9c2hhKG91dHB1dC8ncmVzdW1lLnB0JyksIGNvbnRyYWN0PWlkZW50aXR5LAogICAgICAgIGJpdHdpc2VfZXF1aXZhbGVuY2U9RmFsc2UsIG5vdGU9J0Vwb2NoIGJvdW5kYXJ5IHJlc3VtZTsgZGF0YSBsb2FkZXIgaXMgcmVjb25zdHJ1Y3RlZC4nKQogICAgKG91dHB1dC8nbWFuaWZlc3QuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhtZXRhLCBpbmRlbnQ9MiksIGVuY29kaW5nPSd1dGYtOCcpCiAgICBidW5kbGUgPSBvdXRwdXQvJ2xhdGVzdF9yZWNvdmVyeS56aXAnCiAgICB0ZW1wX2J1bmRsZSA9IG91dHB1dC8nbGF0ZXN0X3JlY292ZXJ5LnppcC50bXAnCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZSh0ZW1wX2J1bmRsZSwgJ3cnLCBjb21wcmVzc2lvbj16aXBmaWxlLlpJUF9TVE9SRUQpIGFzIHo6CiAgICAgICAgei53cml0ZShvdXRwdXQvJ3Jlc3VtZS5wdCcsICdyZXN1bWUucHQnKQogICAgICAgIHoud3JpdGUob3V0cHV0LydtYW5pZmVzdC5qc29uJywgJ21hbmlmZXN0Lmpzb24nKQogICAgICAgIGZvciBuYW1lIGluICgncmVzdWx0cy5jc3YnLCAnYXJncy55YW1sJyk6CiAgICAgICAgICAgIHAgPSBQYXRoKHRyYWluZXIuc2F2ZV9kaXIpL25hbWUKICAgICAgICAgICAgaWYgcC5pc19maWxlKCk6CiAgICAgICAgICAgICAgICB6LndyaXRlKHAsIG5hbWUpCiAgICB0ZW1wX2J1bmRsZS5yZXBsYWNlKGJ1bmRsZSkKICAgIGlmIGVwb2NoID09IDEgb3IgZXBvY2ggJSA1ID09IDAgb3IgZXBvY2ggPT0gdHJhaW5lci5hcmdzLmVwb2NoczoKICAgICAgICBuYW1lZCA9IG91dHB1dC9mJ2Vwb2NoX3tlcG9jaDowM2R9X3JlY292ZXJ5LnppcCcKICAgICAgICBpZiBuYW1lZC5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgRmlsZUV4aXN0c0Vycm9yKG5hbWVkKQogICAgICAgIHNodXRpbC5jb3B5MihidW5kbGUsIG5hbWVkKQogICAgd2l0aCAob3V0cHV0LydjaGVja3BvaW50X2hpc3RvcnkuanNvbmwnKS5vcGVuKCdhJywgZW5jb2Rpbmc9J3V0Zi04JykgYXMgZjoKICAgICAgICBmLndyaXRlKGpzb24uZHVtcHMoZGljdChlcG9jaD1lcG9jaCwgc2hhMjU2PW1ldGFbJ2NoZWNrcG9pbnRfc2hhMjU2J10pKSsnXG4nKQogICAgcHJpbnQoZidCVDFfUkVDT1ZFUllfU0FWRUQgZXBvY2g9e2Vwb2NofSBwYXRoPXtidW5kbGV9JywgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiBtZXRhCgoKZGVmIHJlYWRfcmVjb3ZlcnkoYnVuZGxlLCBkZXN0aW5hdGlvbiwgaWRlbnRpdHkpOgogICAgaW1wb3J0IHRvcmNoCiAgICBkZXN0aW5hdGlvbiA9IFBhdGgoZGVzdGluYXRpb24pCiAgICBkZXN0aW5hdGlvbi5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPUZhbHNlKQogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUoYnVuZGxlKSBhcyB6OgogICAgICAgIGlmIGxlbih6Lm5hbWVsaXN0KCkpICE9IGxlbihzZXQoei5uYW1lbGlzdCgpKSkgb3Igc2V0KHoubmFtZWxpc3QoKSkteydyZXN1bWUucHQnLCdtYW5pZmVzdC5qc29uJywncmVzdWx0cy5jc3YnLCdhcmdzLnlhbWwnfToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVW5leHBlY3RlZCByZWNvdmVyeSBtZW1iZXJzJykKICAgICAgICBmb3IgbmFtZSBpbiB6Lm5hbWVsaXN0KCk6CiAgICAgICAgICAgIGlmIHouZ2V0aW5mbyhuYW1lKS5maWxlX3NpemUgPiAxMDI0KiozOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignT3ZlcnNpemVkIHJlY292ZXJ5IG1lbWJlcicpCiAgICAgICAgICAgIChkZXN0aW5hdGlvbi9uYW1lKS53cml0ZV9ieXRlcyh6LnJlYWQobmFtZSkpCiAgICBtZXRhID0ganNvbi5sb2FkcygoZGVzdGluYXRpb24vJ21hbmlmZXN0Lmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGlmIG1ldGFbJ2NvbnRyYWN0J10gIT0gaWRlbnRpdHkgb3Igc2hhKGRlc3RpbmF0aW9uLydyZXN1bWUucHQnKSAhPSBtZXRhWydjaGVja3BvaW50X3NoYTI1NiddOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1JlY292ZXJ5IGlkZW50aXR5L2hhc2ggbWlzbWF0Y2gnKQogICAgIyBPbmx5IHVzZSBidW5kbGVzIHByb2R1Y2VkIGJ5IHRoaXMgcHJvamVjdDsgYSBoYXNoIGFsb25lIGlzIG5vdCB0cnVzdCBpbiBhIGZvcmVpZ24gcGlja2xlLgogICAgY2twdCA9IHRvcmNoLmxvYWQoZGVzdGluYXRpb24vJ3Jlc3VtZS5wdCcsIG1hcF9sb2NhdGlvbj0nY3B1Jywgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgaWYgKGNrcHQuZ2V0KCdidDFfZm9ybWF0JykgIT0gMSBvciBja3B0LmdldCgnYnQxX2NvbnRyYWN0JykgIT0gaWRlbnRpdHkKICAgICAgICAgICAgb3IgY2twdC5nZXQoJ29wdGltaXplcicpIGlzIE5vbmUgb3IgJ2J0MV9saXZlJyBub3QgaW4gY2twdAogICAgICAgICAgICBvciBpbnQoY2twdFsnZXBvY2gnXSkrMSAhPSBtZXRhWydjb21wbGV0ZWRfZXBvY2hzJ10KICAgICAgICAgICAgb3Igbm90IDAgPCBtZXRhWydjb21wbGV0ZWRfZXBvY2hzJ10gPCBpZGVudGl0eVsnYXJncyddWydlcG9jaHMnXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUmVjb3ZlcnkgaXMgaW5jb21wbGV0ZSwgZmluaXNoZWQsIG9yIGluY29tcGF0aWJsZScpCiAgICByZXR1cm4gY2twdCwgbWV0YQo='}, 'baseline_training_spec.json': {'sha256': 'ba8bcdb26515aadd91ce7aa3c6ccae4dd0eaa38fedab86e23d4e3d903dc316d0', 'base64': 'ew0KICAicHJvdG9jb2wiOiAiQlQtMSIsDQogICJkYXRlIjogIjIwMjYtMDktMTIiLA0KICAic3RhdHVzIjogIkxPQ0FMX1NNT0tFX1BBU1NfS0FHR0xFX1BIT05FX1ZFUklGSUNBVElPTl9CTE9DS0VEIiwNCiAgInB1cnBvc2UiOiAiT25lIHRyYWNlYWJsZSBvcmRpbmFyeSBZT0xPMTFuIGNoZWNrcG9pbnQgZm9yIHdlYWstcmVzcG9uc2UgY2FuZGlkYXRlIGRpYWdub3NpcyIsDQogICJub3RfYV9sYXVuY2hfZmlsZSI6IHRydWUsDQogICJ1bHRyYWx5dGljc19jb21taXQiOiAiMDc5NThhNzAyMDVkMTM4ODYxMmJkMDBmOGEyZjMyY2Y3NjlkOGZlZCIsDQogICJlbnZpcm9ubWVudCI6IHsNCiAgICAicHl0aG9uIjogIjMuMTIiLA0KICAgICJ0b3JjaCI6ICIyLjcuMSIsDQogICAgInRvcmNodmlzaW9uIjogIjAuMjIuMSIsDQogICAgInRvcmNoX3doZWVsX2luZGV4IjogImh0dHBzOi8vZG93bmxvYWQucHl0b3JjaC5vcmcvd2hsL2N1MTI2IiwNCiAgICAiaW5zdGFsbGVkX2FuZF90ZXN0ZWQiOiB0cnVlLA0KICAgICJ2ZXJpZmljYXRpb25fc2NvcGUiOiAiV2luZG93cyBDb25kYSBsb2NhbCBzbW9rZSBvbmx5OyBjbG91ZCBlbnZpcm9ubWVudCBub3QgdGVzdGVkIg0KICB9LA0KICAiaW5pdGlhbGl6YXRpb24iOiB7DQogICAgInVybCI6ICJodHRwczovL2dpdGh1Yi5jb20vdWx0cmFseXRpY3MvYXNzZXRzL3JlbGVhc2VzL2Rvd25sb2FkL3Y4LjMuMC95b2xvMTFuLnB0IiwNCiAgICAiYXNzZXRfaWQiOiAxOTYyMDEyNzYsDQogICAgImV4cGVjdGVkX2J5dGVzIjogNTYxMzc2NCwNCiAgICAic2hhMjU2IjogIjBlYmJjODBkNGE3NjgwZDE0OTg3YTU3N2NkMjEzNDJiNjVlY2ZkOTQ2MzJiZDlhOGRhNjNhZTY0MTc2NDRlZTEiLA0KICAgICJ1c2UiOiAiQ09DTyBpbml0aWFsaXphdGlvbiBvbmx5OyBub3QgYSBWaXNEcm9uZSBjaGVja3BvaW50Ig0KICB9LA0KICAiZGF0YSI6IHsNCiAgICAidHJhaW5fY291bnQiOiA2NDcxLA0KICAgICJ0cmFpbl96aXBfc2hhMjU2IjogIjg2YTc3ZWJhOTMxMzdiZmMxNmU0OTkzODYwZGU5MjQ1YjA2NzVjMGRiYTBkM2FiOThmYjQ1ODY5OWUyNTZmODQiLA0KICAgICJ2YWxfemlwX3NoYTI1NiI6ICJhYmVlYTA2MzAzN2U1ZDIwMzk4ODM3ZGViMTEwODRlNjUyNDAyYTM0ZGRmNGYyMDdiZGY1NDFhNmYyYTM1ZWY5IiwNCiAgICAiY2FsNDhfc29ydCI6ICJhc2NlbmRpbmcgU0hBMjU2KFVURjgoJ2RpYWctdjE6JyArIGltYWdlX2Jhc2VuYW1lKSk7IHRpZSBieSBVbmljb2RlIGJhc2VuYW1lIiwNCiAgICAibmF0aXZlX3ZhbF9jb3VudCI6IDQ4LA0KICAgICJkaWFnNTAwX2FjY2VzcyI6IGZhbHNlLA0KICAgICJ0ZXN0X2Rldl9hY2Nlc3MiOiBmYWxzZSwNCiAgICAidHJhaW5pbmdfcmVnaW9uX2lnbm9yZSI6IGZhbHNlLA0KICAgICJuYW1lcyI6IFsNCiAgICAgICJwZWRlc3RyaWFuIiwNCiAgICAgICJwZW9wbGUiLA0KICAgICAgImJpY3ljbGUiLA0KICAgICAgImNhciIsDQogICAgICAidmFuIiwNCiAgICAgICJ0cnVjayIsDQogICAgICAidHJpY3ljbGUiLA0KICAgICAgImF3bmluZy10cmljeWNsZSIsDQogICAgICAiYnVzIiwNCiAgICAgICJtb3RvciINCiAgICBdDQogIH0sDQogICJ0cmFpbl9hcmdzIjogew0KICAgICJ0YXNrIjogImRldGVjdCIsDQogICAgIm1vZGUiOiAidHJhaW4iLA0KICAgICJlcG9jaHMiOiAxMDAsDQogICAgInRpbWUiOiBudWxsLA0KICAgICJwYXRpZW5jZSI6IDAsDQogICAgImJhdGNoIjogNCwNCiAgICAiaW1nc3oiOiA2NDAsDQogICAgImRldmljZSI6ICIwIiwNCiAgICAid29ya2VycyI6IDAsDQogICAgInNlZWQiOiAwLA0KICAgICJkZXRlcm1pbmlzdGljIjogdHJ1ZSwNCiAgICAiYW1wIjogZmFsc2UsDQogICAgImNhY2hlIjogZmFsc2UsDQogICAgImNvbXBpbGUiOiBmYWxzZSwNCiAgICAib3B0aW1pemVyIjogIlNHRCIsDQogICAgImxyMCI6IDAuMDEsDQogICAgImxyZiI6IDAuMDEsDQogICAgIm1vbWVudHVtIjogMC45MzcsDQogICAgIndlaWdodF9kZWNheSI6IDAuMDAwNSwNCiAgICAid2FybXVwX2Vwb2NocyI6IDMuMCwNCiAgICAid2FybXVwX21vbWVudHVtIjogMC44LA0KICAgICJ3YXJtdXBfYmlhc19sciI6IDAuMSwNCiAgICAibmJzIjogNjQsDQogICAgImNvc19sciI6IGZhbHNlLA0KICAgICJjbHNfcmVtYXAiOiBmYWxzZSwNCiAgICAiZnJlZXplIjogbnVsbCwNCiAgICAicmVzdW1lIjogZmFsc2UsDQogICAgImZyYWN0aW9uIjogMS4wLA0KICAgICJyZWN0IjogZmFsc2UsDQogICAgIm11bHRpX3NjYWxlIjogMC4wLA0KICAgICJzaW5nbGVfY2xzIjogZmFsc2UsDQogICAgImNsc19wdyI6IDAuMCwNCiAgICAiZGlzdGlsbF9tb2RlbCI6IG51bGwsDQogICAgImJveCI6IDcuNSwNCiAgICAiY2xzIjogMC41LA0KICAgICJkZmwiOiAxLjUsDQogICAgImhzdl9oIjogMC4wMTUsDQogICAgImhzdl9zIjogMC43LA0KICAgICJoc3ZfdiI6IDAuNCwNCiAgICAiZGVncmVlcyI6IDAuMCwNCiAgICAidHJhbnNsYXRlIjogMC4xLA0KICAgICJzY2FsZSI6IDAuNSwNCiAgICAic2hlYXIiOiAwLjAsDQogICAgInBlcnNwZWN0aXZlIjogMC4wLA0KICAgICJmbGlwdWQiOiAwLjAsDQogICAgImZsaXBsciI6IDAuNSwNCiAgICAiYmdyIjogMC4wLA0KICAgICJtb3NhaWMiOiAxLjAsDQogICAgImNsb3NlX21vc2FpYyI6IDEwLA0KICAgICJtaXh1cCI6IDAuMCwNCiAgICAiY3V0bWl4IjogMC4wLA0KICAgICJjb3B5X3Bhc3RlIjogMC4wLA0KICAgICJ2YWwiOiB0cnVlLA0KICAgICJzcGxpdCI6ICJ2YWwiLA0KICAgICJjb25mIjogMC4wMDEsDQogICAgImlvdSI6IDAuNSwNCiAgICAibWF4X2RldCI6IDUwMCwNCiAgICAic2F2ZSI6IHRydWUsDQogICAgInNhdmVfcGVyaW9kIjogMTAsDQogICAgInNhdmVfanNvbiI6IGZhbHNlLA0KICAgICJwbG90cyI6IGZhbHNlLA0KICAgICJwcm9maWxlIjogZmFsc2UsDQogICAgImV4aXN0X29rIjogZmFsc2UNCiAgfSwNCiAgInNlbGVjdGVkX2NoZWNrcG9pbnQiOiB7DQogICAgImVwb2NoXzFfYmFzZWQiOiAxMDAsDQogICAgImZpbGUiOiAibGFzdC5wdCIsDQogICAgIndlaWdodHMiOiAiRU1BIiwNCiAgICAic2VsZWN0aW9uX2J5X21ldHJpY3MiOiBmYWxzZSwNCiAgICAicHJlc2VydmVfcHJlX3N0cmlwX29yaWdpbmFsIjogdHJ1ZQ0KICB9LA0KICAiZ3VhcmRzIjogew0KICAgICJhYm9ydF9vbl9vb21fYmVmb3JlX2F1dG9fcmV0cnkiOiB0cnVlLA0KICAgICJhYm9ydF9vbl9ub25maW5pdGUiOiB0cnVlLA0KICAgICJhYm9ydF9vbl9wYXJhbWV0ZXJfZHJpZnQiOiB0cnVlLA0KICAgICJhdXRvbWF0aWNfcmV0cnlfb3JfcmVzdW1lIjogZmFsc2UsDQogICAgImltcGxlbWVudGVkIjogdHJ1ZSwNCiAgICAidmVyaWZpY2F0aW9uX3Njb3BlIjogIk5vcm1hbCB0cmFpbmluZyBwYXRoIGFuZCBzdGF0aWMgcmV2aWV3OyBubyBPT00vTmFOIGZhdWx0IGluamVjdGlvbiINCiAgfSwNCiAgImxpbWl0cyI6IHsNCiAgICAicHJlcGFyYXRpb25faG91cnMiOiAyLA0KICAgICJzYW5pdHlfbWludXRlcyI6IDEwLA0KICAgICJzYW5pdHlfbWF4X2JhdGNoZXMiOiA0LA0KICAgICJ0cmFpbmluZ193YWxsX2hvdXJzIjogMjQsDQogICAgIm91dHB1dF9jaGVja19taW51dGVzIjogMTUsDQogICAgIm5ld19maWxlc19naWIiOiA0MCwNCiAgICAibWluaW11bV9mcmVlX2Rpc2tfZ2liIjogNTANCiAgfSwNCiAgIm91dHB1dF9jaGVjayI6IHsNCiAgICAiZGF0YSI6ICJmaXJzdCAxMCBjYWw0OCBmaWxlcyBpbiBmaXhlZCBvcmRlciIsDQogICAgImltZ3N6IjogNjQwLA0KICAgICJiYXRjaCI6IDEsDQogICAgInByZWNpc2lvbiI6ICJGUDMyIiwNCiAgICAicmVnaW9uX3NlbGVjdGlvbl9vcl9zbGljaW5nIjogZmFsc2UNCiAgfSwNCiAgInBlbmRpbmdfZXhlY3V0aW9uX2ZpZWxkcyI6IFsNCiAgICAiY2xvdWRfYWNjb3VudF9jb25uZWN0aW9uIiwNCiAgICAiY2xvdWRfcnVuX2lkIiwNCiAgICAiY2xvdWRfZW52aXJvbm1lbnRfbG9jayIsDQogICAgImZ1bGxfdHJhaW5fbWFuaWZlc3RfYW5kX2NvbnZlcnNpb24iLA0KICAgICJjbG91ZF9vdXRwdXRfY2hlY2twb2ludF9oYXNoIg0KICBdLA0KICAidXNlcl9hbWVuZG1lbnQiOiB7DQogICAgImRhdGUiOiAiMjAyNi0wOS0xMiIsDQogICAgImVudmlyb25tZW50IjogIkNvbmRhIEg6L0NvbmRhL2VudnMvVUFWX0JUMSIsDQogICAgImxvY2FsX3Ntb2tlIjogew0KICAgICAgImVwb2NocyI6IDEsDQogICAgICAiYmF0Y2giOiAxLA0KICAgICAgInRyYWluX2ltYWdlcyI6IDQNCiAgICB9LA0KICAgICJmdWxsX3RyYWluaW5nX2xvY2F0aW9uIjogIkthZ2dsZSBmaXJzdDsgQ29sYWIgb25seSBhZnRlciBLYWdnbGUgcXVvdGEgZXhoYXVzdGVkIGFuZCBwcmV2aW91cyBzZXNzaW9uIHN0b3BwZWQiLA0KICAgICJwYWlkX2NvbXB1dGVfYXV0aG9yaXplZCI6IGZhbHNlDQogIH0sDQogICJsb2NhbF9zbW9rZV9yZXN1bHQiOiB7DQogICAgInJ1bl9pZCI6ICJCVDEtU01PS0UtMjAyNjA5MTItMDEiLA0KICAgICJjb21wbGV0ZWRfZXBvY2hzIjogMSwNCiAgICAiYmF0Y2giOiAxLA0KICAgICJiYXNlbGluZV9lbGlnaWJsZSI6IGZhbHNlLA0KICAgICJjaGVja3BvaW50X3NoYTI1NiI6ICIyN2Q1NWNlNGQ1ZmFmNWRiMTEyMjU4MzY0ZTgwNTQxMDE0OTBiMGJmNDg0ZTgyMTcyZmRjNDgxNzQ1NDlkMjYzIg0KICB9LA0KICAiY2hlY2twb2ludF9yZXN1bWUiOiB7DQogICAgInVzZXJfYXV0aG9yaXplZCI6IHRydWUsDQogICAgImVwb2NoX2JvdW5kYXJ5IjogdHJ1ZSwNCiAgICAic2VnbWVudF9taW51dGVzIjogNjAsDQogICAgIm51bWJlcmVkX2NvcHlfZXZlcnlfZXBvY2hzIjogNSwNCiAgICAidG90YWxfZXBvY2hzIjogMTAwLA0KICAgICJjdW11bGF0aXZlX3RyYWluaW5nX3dhbGxfaG91cnMiOiAyNCwNCiAgICAic2FtZV9iYXRjaF9hbmRfcHJvdG9jb2xfcmVxdWlyZWQiOiB0cnVlLA0KICAgICJiaXR3aXNlX2VxdWl2YWxlbmNlIjogZmFsc2UsDQogICAgInBsYXRmb3JtX291dHB1dF9iYWNrdXBfcmVxdWlyZWQiOiB0cnVlLA0KICAgICJzZXJpYWxpemF0aW9uX3Rlc3RzIjogIlBBU1M7IG5vIFlPTE8gcmVzdW1lIHRyYWluaW5nIGV4ZWN1dGVkIg0KICB9DQp9DQo='}}
CODE = WORK/'code'
CODE.mkdir()
for name, item in BUNDLE.items():
    raw = base64.b64decode(item['base64'])
    assert hashlib.sha256(raw).hexdigest() == item['sha256']
    (CODE/name).write_bytes(raw)
print('Verified source files:', list(BUNDLE))


In [ ]:
import urllib.request
RAW = WORK / 'raw'; RAW.mkdir()
RESOURCES = [
 ('VisDrone2019-DET-train.zip','https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip','86a77eba93137bfc16e4993860de9245b0675c0dba0d3ab98fb458699e256f84'),
 ('VisDrone2019-DET-val.zip','https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip','abeea063037e5d20398837deb11084e652402a34ddf4f207bdf541a6f2a35ef9'),
 ('yolo11n.pt','https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt','0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1')]
started = time.monotonic()
for name, url, expected in RESOURCES:
    target = RAW/name
    if target.exists(): raise FileExistsError(target)
    digest = hashlib.sha256()
    with urllib.request.urlopen(url, timeout=60) as response, target.with_suffix(target.suffix+'.part').open('xb') as f:
        while chunk := response.read(1024*1024):
            if time.monotonic()-started > 3600: raise TimeoutError('Resource download time limit')
            digest.update(chunk); f.write(chunk)
    if digest.hexdigest() != expected:
        raise RuntimeError('Resource identity changed; do not substitute another dataset: '+name)
    target.with_suffix(target.suffix+'.part').rename(target)
    print(name, 'SHA256 verified')
DATA = WORK / 'prepared'
run([PY,CODE/'prepare_bt1_data.py','--train-zip',RAW/'VisDrone2019-DET-train.zip','--val-zip',RAW/'VisDrone2019-DET-val.zip','--output',DATA], 1800, 'prepare_data.log')
size = sum(p.stat().st_size for p in WORK.rglob('*') if p.is_file())
if size > 40*1024**3: raise RuntimeError('40 GiB artifact budget exceeded')
print(json.loads((DATA/'preparation.json').read_text())['counts'])


## 启动或接续一段普通基线
训练目标100轮，约60分钟后于轮末暂停；不是把学习率计划改成短训练。首次从COCO初始化，后续载入恢复包。段末必须保存平台输出并取得ZIP，不同时启动其他平台。


In [ ]:
if not isinstance(CLOUD_BATCH, int) or CLOUD_BATCH < 1:
    raise ValueError('CLOUD_BATCH must be a fixed positive integer')
RUN = RUN_PARENT / ('BT1-CLOUD-' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()))
if RUN.exists(): raise FileExistsError(RUN)
(WORK/'cloud_request.json').write_text(json.dumps({'batch':CLOUD_BATCH,'epochs':100,'run':str(RUN),'started_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'purpose':'ordinary baseline only'},indent=2))
command = [PY,'-u',CODE/'run_bt1.py','--data',DATA/'data.yaml','--weights',RAW/'yolo11n.pt','--spec',CODE/'baseline_training_spec.json','--output',RUN,'--batch',str(CLOUD_BATCH),'--segment-minutes','60']
if RESUME_BUNDLE:
    command += ['--resume-bundle',str(RESUME_BUNDLE)]
run(command, 24*3600, 'cloud_train.log')
status = json.loads((RUN/'run_status.json').read_text())
assert status['status'] in ('PASS','PAUSED')
if status['status']=='PASS': assert status['completed_epochs']==100
print(status)


In [ ]:
# Run this after completion, or after a reported failure while this session still exists.
# Includes only execution evidence and weights, not the large datasets or virtual environment.
import zipfile
archive = RUN_PARENT / (RUN.name + '_evidence.zip')
with zipfile.ZipFile(archive,'x',compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN.rglob('*'):
        if p.is_file(): z.write(p, RUN.name+'/'+str(p.relative_to(RUN)))
    for name in ['pip_freeze.txt','cloud_request.json','cloud_train.log','pip_torch_report.json','pip_ultralytics_report.json','prepare_data.log']:
        if (WORK/name).is_file(): z.write(WORK/name,'provenance/'+name)
    for p in CODE.iterdir():
        if p.is_file(): z.write(p,'code/'+p.name)
    for name in ['preparation.json','train.txt','cal48.txt','diag500_names.txt','conversion.jsonl']:
        z.write(DATA/name,'data_provenance/'+name)
print('Save/download this file through the platform:', archive)
print('No candidate performance or official evaluation has been run.')
